In [1]:
import pandas as pd

In [73]:
"""initail data and stuff"""

divisions = {
    "AL-East":    ("BAL", "BOS", "NYA", "TBA", "TOR"),
    "AL-Central": ("CHA", "CLE", "DET", "KCA", "MIN"),
    "AL-West":    ("ANA", "HOU", "OAK", "SEA", "TEX"),

    "NL-East":    ("ATL", "MIA", "NYN", "PHI", "WAS"),
    "NL-Central": ("CHN", "CIN", "MIL", "PIT", "SLN"),
    "NL-West":    ("ARI", "COL", "LAN", "SDN", "SFN")
}

**PART 1 CODE**

In [ ]:
"""PART 1 CODE (for 2014+)"""

rows = []

for season in range(2014, 2025):
    for division, teams in divisions.items():
        rows.append({
            "season": season,
            "league": "MLB",
            "division": division,
            "division_teams": teams
        })

division_df = pd.DataFrame(rows)
division_df.to_csv("mlb_divisions.csv", index=False)

**PART 2 CODE**

In [75]:
games_df = pd.read_csv("game_data_us_leagues.csv")

# Make one row per season/team/division
team_division_df = (
    division_df
    .explode("division_teams")
    .rename(columns={"division_teams": "team"})
)

# this is only the baseball games
mlb_games = games_df[
    (games_df["league"] == "MLB") &
    (games_df["season"].between(2014, 2024))
].copy()

In [76]:
# adding team 1 and team 2 division by merging with our mlb_divisions
mlb_games = mlb_games.merge(
    team_division_df[["season", "team", "division"]],
    left_on=["season", "team1"],
    right_on=["season", "team"],
    how="left"
)

mlb_games = (
    mlb_games
    .rename(columns={"division": "division1"})
    .drop(columns="team")
)

# adds team 2 div
mlb_games = mlb_games.merge(
    team_division_df[["season", "team", "division"]],
    left_on=["season", "team2"],
    right_on=["season", "team"],
    how="left"
)

mlb_games = (
    mlb_games
    .rename(columns={"division": "division2"})
    .drop(columns="team")
)

# season team1 team2 division1 division2

In [77]:
# FILTERING BY DIVISON
div_games = mlb_games[
    mlb_games["division1"] == mlb_games["division2"]
].copy()

div_games["division"] = div_games["division1"]

div_games = div_games.drop(columns=["division1", "division2"])
#div_games.head()

In [78]:
# go game by game and figure out the winner
def get_winner(row):
    if row["score1"] > row["score2"]:
        return row["team1"]
    elif row["score2"] > row["score1"]:
        return row["team2"]
    else:
        return None


div_games["winner"] = div_games.apply(
    get_winner,
    axis=1
)

In [79]:
# pairs need to have same ordering so wehn we do pd.group we dont double count
teams = div_games[["team1", "team2"]]

div_games["team1"] = teams.min(axis=1)
div_games["team2"] = teams.max(axis=1)

In [80]:
# mark whoever winner in a div game
div_games["win1"] = (
    div_games["winner"] == div_games["team1"]
).astype(int)

div_games["win2"] = (
    div_games["winner"] == div_games["team2"]
).astype(int)

In [81]:
# compute the series games
series_df = (
    div_games
    .groupby(
        ["season", "division", "team1", "team2"],
        as_index=False
    )
    .agg(
        wins1=("win1", "sum"),
        wins2=("win2", "sum")
    )
)

In [82]:
# determine the full series winner
def get_series_result(row):
    if row["wins1"] > row["wins2"]:
        return 1
    elif row["wins2"] > row["wins1"]:
        return -1
    else:
        return 0


series_df["result"] = series_df.apply(
    get_series_result,
    axis=1
)

# adds the other column we need to specifiy that this is mlb
series_df.insert(
    1,
    "league",
    "MLB"
)

In [83]:
series_df = series_df[
    [
        "season",
        "league",
        "division",
        "team1",
        "team2",
        "result",
        "wins1",
        "wins2"
    ]
]

In [90]:
series_df.head()

,season,league,division,team1,team2,result,wins1,wins2
0,2014,MLB,AL-Central,CHA,CLE,-1,9,10
1,2014,MLB,AL-Central,CHA,DET,-1,9,10
2,2014,MLB,AL-Central,CHA,KCA,-1,6,13
3,2014,MLB,AL-Central,CHA,MIN,-1,9,10
4,2014,MLB,AL-Central,CLE,DET,-1,8,11
